<a href="https://colab.research.google.com/github/MalakSalehh/Thesis-datasets/blob/main/ml2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install xgboost scikit-learn pandas numpy scipy

In [2]:
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, cross_val_predict, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error, mean_squared_error, cohen_kappa_score
from scipy.stats import pearsonr
from scipy.sparse import hstack, csr_matrix

from xgboost import XGBRegressor

In [3]:
import pandas as pd

url = "https://raw.githubusercontent.com/MalakSalehh/Thesis-datasets/main/training_set_rel3.tsv"

df = pd.read_csv(url, sep="\t", encoding="latin1")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (12976, 28)


,essay_id,essay_set,essay,rater1_domain1,rater2_domain1,rater3_domain1,domain1_score,rater1_domain2,rater2_domain2,domain2_score,...,rater2_trait3,rater2_trait4,rater2_trait5,rater2_trait6,rater3_trait1,rater3_trait2,rater3_trait3,rater3_trait4,rater3_trait5,rater3_trait6
0,1,1,"Dear local newspaper, I think effects computer...",4,4,NaN,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",5,4,NaN,9,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",4,3,NaN,7,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",5,5,NaN,10,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1,"Dear @LOCATION1, I know having computers has a...",4,4,NaN,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
set1 = df[df['essay_set'] == 1].copy()
set1.shape

(1783, 28)

# **CLEANING**

In [13]:
def clean_text(text):
    text = str(text)
    text = text.replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

set1['essay_clean'] = set1['essay'].apply(clean_text)

Linguistic features:
- In addition to textual representations, simple handcrafted linguistic features were extracted to capture structural and stylistic properties of the essays. These included word count, sentence count, and average word length. Word count reflects the overall length and development of the essay, while sentence count provides an indication of structural organization. Average word length serves as a proxy for vocabulary complexity. These features are commonly used in automated essay scoring systems as they provide useful signals about writing quality, despite not capturing deeper semantic meaning.

In [14]:
def word_count(text):
    return len(text.split())

def sentence_count(text):
    sentences = re.split(r'[.!?]+', text)
    sentences = [s for s in sentences if s.strip()]
    return len(sentences)

def avg_word_len(text):
    words = text.split()
    if not words:
        return 0
    return np.mean([len(w) for w in words])

set1['word_count'] = set1['essay_clean'].apply(word_count)
set1['sentence_count'] = set1['essay_clean'].apply(sentence_count)
set1['avg_word_len'] = set1['essay_clean'].apply(avg_word_len)

define x and y


In [15]:
X_text = set1["essay_clean"]
X_extra = set1[["word_count", "sentence_count", "avg_word_len"]]
y = set1["domain1_score"].astype(int)
essay_ids = set1["essay_id"]

In [17]:
print(set1.shape)
print(set1["domain1_score"].value_counts().sort_index())
print(set1["domain1_score"].isna().sum())

(1783, 32)
domain1_score
2      10
3       1
4      17
5      17
6     110
7     135
8     687
9     334
10    316
11    109
12     47
Name: count, dtype: int64
0


In [18]:
set1 = set1[set1["domain1_score"] != 3] ##to stratify since score 3 is only found once

In [19]:
X_text = set1["essay_clean"]
X_extra = set1[["word_count", "sentence_count", "avg_word_len"]]
y = set1["domain1_score"].astype(int)
essay_ids = set1["essay_id"]

In [20]:
##Split before TF-IDF
from sklearn.model_selection import train_test_split

from sklearn.model_selection import train_test_split

X_text_train, X_text_test, X_extra_train, X_extra_test, y_train, y_test, id_train, id_test = train_test_split(
    X_text,
    X_extra,
    y,
    essay_ids,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Create TF-IDF features (It is a way to turn essay text into numbers so the ML model can use it.)

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test = tfidf.transform(X_text_test)

X_extra_train_sparse = csr_matrix(X_extra_train.values)
X_extra_test_sparse = csr_matrix(X_extra_test.values)

X_train = hstack([X_tfidf_train, X_extra_train_sparse])
X_test = hstack([X_tfidf_test, X_extra_test_sparse])

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (1425, 5003)
X_test shape: (357, 5003)


**Combine TF-IDF with linguistic features**


**Train a baseline XGBoost model**

In [22]:

from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)

**Predict**

In [23]:
y_pred = xgb_model.predict(X_test)

y_pred_round = np.rint(y_pred).astype(int)
y_pred_round = np.clip(y_pred_round, 2, 12)

In [24]:
qwk = cohen_kappa_score(y_test, y_pred_round, weights='quadratic')
kappa = cohen_kappa_score(y_test, y_pred_round)
pcc, _ = pearsonr(y_test, y_pred_round)
mae = mean_absolute_error(y_test, y_pred_round)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_round))

print("QWK:", qwk)
print("Cohen's Kappa:", kappa)
print("PCC:", pcc)
print("MAE:", mae)
print("RMSE:", rmse)

QWK: 0.8403913956344333
Cohen's Kappa: 0.39269115100701735
PCC: 0.8466432468897023
MAE: 0.5350140056022409
RMSE: 0.8147794463647738


In [25]:
results_df = pd.DataFrame({
    'essay_id': set1.iloc[y_test.index]['essay_id'].values if hasattr(y_test, 'index') else np.nan,
    'human_score': y_test.values,
    'predicted_score': y_pred_round
})

results_df.head(20)

,essay_id,human_score,predicted_score
0,832,8,8
1,445,8,8
2,1228,8,7
3,341,9,9
4,702,8,9
5,1322,9,11
6,51,5,4
7,1133,8,8
8,1560,7,6
9,789,8,8


In [26]:
def score_category(score):
    if 2 <= score <= 5:
        return 'Low'
    elif 6 <= score <= 8:
        return 'Medium'
    else:
        return 'High'

results_df['human_category'] = results_df['human_score'].apply(score_category)
results_df['predicted_category'] = results_df['predicted_score'].apply(score_category)

results_df.head(20)

,essay_id,human_score,predicted_score,human_category,predicted_category
0,832,8,8,Medium,Medium
1,445,8,8,Medium,Medium
2,1228,8,7,Medium,Medium
3,341,9,9,High,High
4,702,8,9,Medium,High
5,1322,9,11,High,High
6,51,5,4,Low,Low
7,1133,8,8,Medium,Medium
8,1560,7,6,Medium,Medium
9,789,8,8,Medium,Medium


**Cross validtation:**
-  Instead of evaluating your model on one test split, it evaluates it multiple times on different splits

In [29]:
cv_data = set1[["essay_clean", "word_count", "sentence_count", "avg_word_len"]].copy()
cv_target = set1["domain1_score"].astype(int)

In [30]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, cohen_kappa_score
from scipy.stats import pearsonr
import numpy as np

preprocessor = ColumnTransformer(
    transformers=[
        ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words="english"), "essay_clean"),
        ("num", "passthrough", ["word_count", "sentence_count", "avg_word_len"])
    ]
)

cv_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    ))
])

In [31]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_preds = cross_val_predict(cv_pipeline, cv_data, cv_target, cv=skf)
cv_preds_round = np.rint(cv_preds).astype(int)
cv_preds_round = np.clip(cv_preds_round, 2, 12)

cv_qwk = cohen_kappa_score(cv_target, cv_preds_round, weights="quadratic")
cv_kappa = cohen_kappa_score(cv_target, cv_preds_round)
cv_pcc, _ = pearsonr(cv_target, cv_preds_round)
cv_mae = mean_absolute_error(cv_target, cv_preds_round)
cv_rmse = np.sqrt(mean_squared_error(cv_target, cv_preds_round))

print("5-Fold CV QWK:", cv_qwk)
print("5-Fold CV Cohen's Kappa:", cv_kappa)
print("5-Fold CV PCC:", cv_pcc)
print("5-Fold CV MAE:", cv_mae)
print("5-Fold CV RMSE:", cv_rmse)

5-Fold CV QWK: 0.8328964592487141
5-Fold CV Cohen's Kappa: 0.3690486017704272
5-Fold CV PCC: 0.8403111300819417
5-Fold CV MAE: 0.5566778900112234
5-Fold CV RMSE: 0.83282812968985
